In [ ]:
# Install required libraries
!pip install -q streamlit pandas numpy scikit-learn xgboost matplotlib seaborn openpyxl
print("Dependencies installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 79.5 MB/s eta 0:00:00
Dependencies installed successfully!


In [ ]:
import os

# Create native Streamlit configuration
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write("""[theme]
base = "dark"
primaryColor = "#06B6D4"
backgroundColor = "#0B0F19"
secondaryBackgroundColor = "#111827"
textColor = "#F9FAFB"
font = "sans serif"

[server]
headless = true
enableCORS = false
enableXsrfProtection = false
""")

print("Obsidian & Electric Cyan theme configured in .streamlit/config.toml!")

Obsidian & Electric Cyan theme configured in .streamlit/config.toml!


In [ ]:
import io
import os
import zipfile
import requests
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import pickle

# 0. Check and download dataset if missing
if not os.path.exists("Online Retail.xlsx"):
    print("Dataset not found locally. Downloading from UCI archive...")
    url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
    resp = requests.get(url)
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        z.extractall()
    print("Download and extraction complete!")

# 1. Load and Clean Dataset
print("Loading Online Retail dataset...")
df = pd.read_excel("Online Retail.xlsx")

# Filter cancellations, missing customers, and non-positive records
df['IsCancelled'] = df['InvoiceNo'].astype(str).str.startswith('C')
df_clean = df[~df['IsCancelled']].dropna(subset=['CustomerID']).copy()
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int).astype(str)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['TotalSales'] = df_clean['Quantity'] * df_clean['UnitPrice']

# 2. Time-Based Cutoff for Forward 90-Day LTV
max_date = df_clean['InvoiceDate'].max()
cutoff_date = max_date - pd.Timedelta(days=90)

obs_df = df_clean[df_clean['InvoiceDate'] <= cutoff_date].copy()
target_df = df_clean[df_clean['InvoiceDate'] > cutoff_date].copy()

# 3. Calculate RFM on Observation Window
rfm = obs_df.groupby('CustomerID').agg({
    'InvoiceDate': lambda d: (cutoff_date - d.max()).days,
    'InvoiceNo': 'nunique',
    'TotalSales': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# Target 90-Day Forward Spend
target_spend = target_df.groupby('CustomerID')['TotalSales'].sum().reset_index()
target_spend.columns = ['CustomerID', 'TargetLTV_90d']

# Merge and create behavioral ratios
data = pd.merge(rfm, target_spend, on='CustomerID', how='left').fillna(0)
data['AOV'] = data['Monetary'] / data['Frequency']

# 4. K-Means Clustering
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(data[['Recency', 'Frequency', 'Monetary']])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
data['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Assign Personas by Mean Monetary Rank
cluster_rank = data.groupby('Cluster')['Monetary'].mean().sort_values().index.tolist()
persona_map = {
    cluster_rank[0]: 'Hibernating / Low Value',
    cluster_rank[1]: 'At-Risk / Occasional',
    cluster_rank[2]: 'Loyal Core Customers',
    cluster_rank[3]: 'High-Value Champions'
}
data['Segment'] = data['Cluster'].map(persona_map)

# 5. Train Supervised XGBoost Regressor
X = data[['Recency', 'Frequency', 'Monetary', 'AOV', 'Cluster']]
y = data['TargetLTV_90d']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluation
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)
print(f"Model evaluation: Test RMSE = ${rmse:.2f}, R² = {r2:.4f}")

# 6. Save Artifacts
data.to_csv("customer_segments.csv", index=False)
with open("rfm_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("kmeans_model.pkl", "wb") as f:
    pickle.dump(kmeans, f)
with open("xgb_ltv_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Pipeline execution complete! Saved dataset and model artifacts.")

Dataset not found locally. Downloading from UCI archive...
Download and extraction complete!
Loading Online Retail dataset...
Model evaluation: Test RMSE = $6901.23, R² = 0.2682
Pipeline execution complete! Saved dataset and model artifacts.


In [ ]:
%%writefile app.py
import os
import pickle
import numpy as np
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import seaborn as sns

st.set_page_config(
    page_title="Customer Persona & LTV Hub",
    page_icon="⚡",
    layout="wide"
)

# Obsidian & Electric Cyan Theme Variables
OBSIDIAN_BG = "#0B0F19"
PANEL_BG = "#111827"
CYAN_ACCENT = "#06B6D4"
PURPLE_ACCENT = "#A855F7"
TEXT_COLOR = "#F9FAFB"
MUTED_GRID = "#1F2937"

sns.set_theme(style="darkgrid")
plt.rcParams['figure.facecolor'] = PANEL_BG
plt.rcParams['axes.facecolor'] = OBSIDIAN_BG
plt.rcParams['text.color'] = TEXT_COLOR
plt.rcParams['axes.labelcolor'] = "#D1D5DB"
plt.rcParams['xtick.color'] = "#9CA3AF"
plt.rcParams['ytick.color'] = "#9CA3AF"
plt.rcParams['grid.color'] = MUTED_GRID

# Custom glowing metric styling
st.markdown(f"""
    <style>
    div[data-testid="stMetricValue"] {{
        color: {CYAN_ACCENT} !important;
        font-weight: 700 !important;
    }}
    div[data-testid="stMetricLabel"] {{
        color: #9CA3AF !important;
    }}
    </style>
""", unsafe_allow_html=True)

@st.cache_data
def load_data():
    if not os.path.exists("customer_segments.csv"):
        return None
    return pd.read_csv("customer_segments.csv")

data = load_data()

if data is None:
    st.error("Missing dataset: customer_segments.csv not found.")
    st.stop()

# Load model weights
with open("rfm_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
with open("kmeans_model.pkl", "rb") as f:
    kmeans = pickle.load(f)
with open("xgb_ltv_model.pkl", "rb") as f:
    xgb_model = pickle.load(f)

# Sidebar
st.sidebar.title("⚡ Segment Filtering")
segments = sorted(data['Segment'].unique().tolist())
selected_segments = st.sidebar.multiselect("Active Segments:", segments, default=segments)
filtered_data = data[data['Segment'].isin(selected_segments)]

st.sidebar.markdown("---")
st.sidebar.caption("System: Predictive LTV & Persona Analytics")
st.sidebar.caption("Theme: Obsidian & Electric Cyan")

# Header & KPIs
st.title("⚡ Customer Intelligence & LTV Portal")
st.markdown("Machine learning clustering combined with 90-day forward spend regression.")

k1, k2, k3, k4 = st.columns(4)
total_count = len(filtered_data)
avg_monetary = filtered_data['Monetary'].mean()
avg_ltv = filtered_data['TargetLTV_90d'].mean()
high_val_share = (len(filtered_data[filtered_data['Segment'] == 'High-Value Champions']) / max(total_count, 1)) * 100

k1.metric("Selected Customers", f"{total_count:,}")
k2.metric("Mean Past Spend", f"${avg_monetary:,.2f}")
k3.metric("Avg Projected 90d LTV", f"${avg_ltv:,.2f}")
k4.metric("Champions Proportion", f"{high_val_share:.1f}%")

st.markdown("---")

tab1, tab2, tab3 = st.tabs(["📊 Cluster Architecture", "📈 Behavioral Distributions", "🔮 LTV Live Scoring"])

with tab1:
    col_left, col_right = st.columns(2)

    with col_left:
        st.subheader("Population by Segment")
        seg_counts = filtered_data['Segment'].value_counts()
        fig, ax = plt.subplots(figsize=(6, 4))
        colors = ["#06B6D4", "#3B82F6", "#8B5CF6", "#A855F7"]
        sns.barplot(x=seg_counts.values, y=seg_counts.index, palette=colors[:len(seg_counts)], ax=ax)
        ax.set_xlabel("Customer Count")
        st.pyplot(fig)

    with col_right:
        st.subheader("Revenue Contribution by Segment ($)")
        seg_rev = filtered_data.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.barplot(x=seg_rev.values, y=seg_rev.index, palette="mako_r", ax=ax)
        ax.set_xlabel("Historical Revenue ($)")
        st.pyplot(fig)

    st.subheader("Detailed Segment Diagnostics")
    diag_table = filtered_data.groupby('Segment').agg({
        'CustomerID': 'count',
        'Recency': 'mean',
        'Frequency': 'mean',
        'Monetary': 'mean',
        'TargetLTV_90d': 'mean'
    }).rename(columns={
        'CustomerID': 'Cohort Size',
        'Recency': 'Mean Inactivity (Days)',
        'Frequency': 'Mean Invoices',
        'Monetary': 'Historical Spend ($)',
        'TargetLTV_90d': 'Projected 90d Spend ($)'
    }).round(2)
    st.dataframe(diag_table, use_container_width=True)

with tab2:
    plot_left, plot_right = st.columns(2)

    with plot_left:
        st.subheader("Order Frequency vs. Lifetime Revenue")
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.scatterplot(
            data=filtered_data,
            x='Frequency',
            y='Monetary',
            hue='Segment',
            palette="cool",
            alpha=0.75,
            ax=ax
        )
        ax.set_xlim(0, 40)
        ax.set_ylim(0, 15000)
        ax.legend(frameon=True, facecolor=PANEL_BG, edgecolor=MUTED_GRID)
        st.pyplot(fig)

    with plot_right:
        st.subheader("Inactivity (Recency) vs. Forecasted 90d LTV")
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.scatterplot(
            data=filtered_data,
            x='Recency',
            y='TargetLTV_90d',
            hue='Segment',
            palette="flare",
            alpha=0.75,
            ax=ax
        )
        ax.set_ylim(0, 5000)
        ax.legend(frameon=True, facecolor=PANEL_BG, edgecolor=MUTED_GRID)
        st.pyplot(fig)

with tab3:
    st.subheader("Instant Customer Value Inference")
    st.markdown("Supply customer purchase signals to classify their persona and predict their value over the next 90 days.")

    in1, in2, in3 = st.columns(3)
    recency_input = in1.number_input("Days Since Last Invoice (Recency):", min_value=0, max_value=365, value=12)
    frequency_input = in2.number_input("Total Successful Invoices (Frequency):", min_value=1, max_value=250, value=6)
    monetary_input = in3.number_input("Total Historical Volume ($ Monetary):", min_value=1.0, max_value=100000.0, value=1150.0)

    if st.button("Run Real-Time Valuation"):
        # Cluster Prediction
        scaled_input = scaler.transform([[recency_input, frequency_input, monetary_input]])
        cluster_pred = kmeans.predict(scaled_input)[0]

        ranked_clusters = data.groupby('Cluster')['Monetary'].mean().sort_values().index.tolist()
        label_lookup = {
            ranked_clusters[0]: 'Hibernating / Low Value',
            ranked_clusters[1]: 'At-Risk / Occasional',
            ranked_clusters[2]: 'Loyal Core Customers',
            ranked_clusters[3]: 'High-Value Champions'
        }
        assigned_label = label_lookup.get(cluster_pred, 'Standard Customer')

        # XGBoost LTV Prediction
        aov_derived = monetary_input / frequency_input
        model_payload = pd.DataFrame([{
            'Recency': recency_input,
            'Frequency': frequency_input,
            'Monetary': monetary_input,
            'AOV': aov_derived,
            'Cluster': cluster_pred
        }])

        predicted_spend = max(0.0, float(xgb_model.predict(model_payload)[0]))

        out1, out2 = st.columns(2)
        out1.info(f"**Assigned Persona:** `{assigned_label}`")
        out2.success(f"**Predicted 90-Day LTV:** `${predicted_spend:,.2f}`")

Writing app.py


In [ ]:
import os
import subprocess
import time
import re

# 1. Install cloudflared if it is not found on the system
if not os.path.exists("/usr/local/bin/cloudflared") and not os.path.exists("/usr/bin/cloudflared"):
    print("Installing Cloudflared...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    print("Cloudflared installed successfully!")

# 2. Terminate any previous instances
!pkill -f streamlit
!pkill -f cloudflared

# 3. Launch Streamlit daemon in the background
!nohup streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false > /dev/null 2>&1 &
time.sleep(3)

# 4. Start Cloudflared tunnel
pipe = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# 5. Extract and print the clickable link
url = None
for _ in range(30):
    line = pipe.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break
    time.sleep(0.5)

if url:
    print("\n" + "=" * 60)
    print(f"👉 CLICK TO OPEN YOUR NEW PORTAL:\n{url}")
    print("=" * 60 + "\n")
else:
    print("Tunnel is initializing. Please re-run this cell once more if the link does not appear.")

Installing Cloudflared...
Cloudflared installed successfully!

👉 CLICK TO OPEN YOUR NEW PORTAL:
https://commons-beaches-obviously-seekers.trycloudflare.com

